# TSGP Transformer Training (Colab)
Upload `training_pairs.pkl` to the Colab runtime before running.

In [ ]:
!pip install tensorflow keras numpy tqdm

In [ ]:
from google.colab import files
import os

if not os.path.exists('training_pairs.pkl'):
    uploaded = files.upload()
    print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# --- Config ---
NUM_FEATURES = 4

PRIMITIVE_SET_FUNCTIONS = ["add", "sub", "mul", "protdiv"]
TERMINAL_VARIABLES = [f"x{i}" for i in range(NUM_FEATURES)]
ERC_MIN = -0.5
ERC_MAX = 0.5
ERC_STEP = 0.1
ERC_VALUES = [round(ERC_MIN + i * ERC_STEP, 1)
              for i in range(int((ERC_MAX - ERC_MIN) / ERC_STEP) + 1)]

TRANSFORMER_NUM_HEADS = 8
TRANSFORMER_HIDDEN_DIM = 128
TRANSFORMER_NUM_ENCODER_LAYERS = 2
TRANSFORMER_NUM_DECODER_LAYERS = 2
TRANSFORMER_MAX_SEQ_LEN = 100
TRANSFORMER_DROPOUT = 0.1
TRANSFORMER_LEARNING_RATE = 1e-3
TRANSFORMER_EPOCHS = 8
TRANSFORMER_BATCH_SIZE = 256

print("Config loaded.")

In [ ]:
# --- Tokenizer ---
import numpy as np

PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN]
FUNCTION_TOKENS = PRIMITIVE_SET_FUNCTIONS
VARIABLE_TOKENS = TERMINAL_VARIABLES
ERC_TOKENS = [str(v) for v in ERC_VALUES]

VOCAB = SPECIAL_TOKENS + FUNCTION_TOKENS + VARIABLE_TOKENS + ERC_TOKENS

TOKEN_TO_ID = {token: idx for idx, token in enumerate(VOCAB)}
ID_TO_TOKEN = {idx: token for idx, token in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)

PAD_ID = TOKEN_TO_ID[PAD_TOKEN]
SOS_ID = TOKEN_TO_ID[SOS_TOKEN]
EOS_ID = TOKEN_TO_ID[EOS_TOKEN]

FUNCTION_IDS = {TOKEN_TO_ID[t] for t in FUNCTION_TOKENS}
TERMINAL_IDS = {TOKEN_TO_ID[t] for t in VARIABLE_TOKENS + ERC_TOKENS}

ARITY = {}
for t in FUNCTION_TOKENS:
    ARITY[TOKEN_TO_ID[t]] = 2
for t in VARIABLE_TOKENS + ERC_TOKENS:
    ARITY[TOKEN_TO_ID[t]] = 0


def encode(tokens, max_len=TRANSFORMER_MAX_SEQ_LEN):
    ids = [TOKEN_TO_ID.get(t, PAD_ID) for t in tokens]
    ids = ids[:max_len]
    return ids + [PAD_ID] * (max_len - len(ids))


def encode_with_sos_eos(tokens, max_len=TRANSFORMER_MAX_SEQ_LEN):
    ids = [SOS_ID] + [TOKEN_TO_ID.get(t, PAD_ID) for t in tokens] + [EOS_ID]
    ids = ids[:max_len]
    return ids + [PAD_ID] * (max_len - len(ids))


def batch_encode_encoder(token_lists, max_len=TRANSFORMER_MAX_SEQ_LEN):
    return np.array([encode(tl, max_len) for tl in token_lists], dtype=np.int32)


def batch_encode_decoder(token_lists, max_len=TRANSFORMER_MAX_SEQ_LEN):
    return np.array([encode_with_sos_eos(tl, max_len) for tl in token_lists], dtype=np.int32)

print(f"Vocab size: {VOCAB_SIZE}")

In [ ]:
# --- Transformer Model ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model
        positions = np.arange(max_len)[:, np.newaxis]
        dims = np.arange(d_model)[np.newaxis, :]
        angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.pe = tf.constant(angles[np.newaxis, :, :].astype(np.float32))

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"max_len": self.max_len, "d_model": self.d_model})
        return cfg


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = keras.Sequential([
            layers.Dense(dff, activation="relu"),
            layers.Dense(d_model),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, padding_mask=None, training=False):
        attn_output = self.mha(x, x, x, attention_mask=padding_mask, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        x = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(x)
        ffn_output = self.dropout2(ffn_output, training=training)
        x = self.layernorm2(x + ffn_output)
        return x


class TransformerDecoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.mha2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = keras.Sequential([
            layers.Dense(dff, activation="relu"),
            layers.Dense(d_model),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.dropout3 = layers.Dropout(dropout_rate)

    def call(self, x, enc_output, look_ahead_mask=None,
             padding_mask=None, training=False):
        attn1 = self.mha1(x, x, x, attention_mask=look_ahead_mask, training=training)
        attn1 = self.dropout1(attn1, training=training)
        x = self.layernorm1(x + attn1)
        attn2 = self.mha2(x, enc_output, enc_output,
                          attention_mask=padding_mask, training=training)
        attn2 = self.dropout2(attn2, training=training)
        x = self.layernorm2(x + attn2)
        ffn_output = self.ffn(x)
        ffn_output = self.dropout3(ffn_output, training=training)
        x = self.layernorm3(x + ffn_output)
        return x


class TSGPTransformer(keras.Model):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=TRANSFORMER_HIDDEN_DIM,
                 num_heads=TRANSFORMER_NUM_HEADS,
                 num_encoder_layers=TRANSFORMER_NUM_ENCODER_LAYERS,
                 num_decoder_layers=TRANSFORMER_NUM_DECODER_LAYERS,
                 dff=None, max_seq_len=TRANSFORMER_MAX_SEQ_LEN,
                 dropout_rate=TRANSFORMER_DROPOUT, **kwargs):
        super().__init__(**kwargs)
        if dff is None:
            dff = d_model * 4
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.vocab_size = vocab_size

        self.enc_embedding = layers.Embedding(vocab_size, d_model)
        self.dec_embedding = layers.Embedding(vocab_size, d_model)
        self.enc_pos_encoding = PositionalEncoding(max_seq_len, d_model)
        self.dec_pos_encoding = PositionalEncoding(max_seq_len, d_model)
        self.sd_projection = layers.Dense(d_model)
        self.enc_dropout = layers.Dropout(dropout_rate)
        self.dec_dropout = layers.Dropout(dropout_rate)

        self.encoder_layers_list = [
            TransformerEncoderBlock(d_model, num_heads, dff, dropout_rate)
            for _ in range(num_encoder_layers)
        ]
        self.decoder_layers_list = [
            TransformerDecoderBlock(d_model, num_heads, dff, dropout_rate)
            for _ in range(num_decoder_layers)
        ]
        self.final_layer = layers.Dense(vocab_size)

    def _create_padding_mask(self, seq):
        mask = tf.cast(tf.not_equal(seq, PAD_ID), tf.float32)
        return tf.expand_dims(tf.expand_dims(mask, 1), 1)

    def encode(self, enc_input, sd_input, training=False):
        enc_padding_mask = self._create_padding_mask(enc_input)
        x = self.enc_embedding(enc_input)
        x = x * np.sqrt(float(self.d_model))
        x = self.enc_pos_encoding(x)
        sd_expanded = tf.expand_dims(tf.expand_dims(sd_input, -1), -1)
        sd_embed = self.sd_projection(sd_expanded)
        sd_embed = tf.broadcast_to(sd_embed, tf.shape(x))
        x = x + sd_embed
        x = self.enc_dropout(x, training=training)
        for enc_layer in self.encoder_layers_list:
            x = enc_layer(x, padding_mask=enc_padding_mask, training=training)
        return x

    def decode(self, dec_input, enc_output, enc_input, sd_input, training=False):
        seq_len = tf.shape(dec_input)[1]
        look_ahead = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        look_ahead = tf.reshape(look_ahead, [1, 1, seq_len, seq_len])
        dec_padding = tf.cast(tf.not_equal(dec_input, PAD_ID), tf.float32)
        dec_padding = tf.expand_dims(tf.expand_dims(dec_padding, 1), 1)
        combined_mask = look_ahead * dec_padding
        enc_padding_mask = self._create_padding_mask(enc_input)
        x = self.dec_embedding(dec_input)
        x = x * np.sqrt(float(self.d_model))
        x = self.dec_pos_encoding(x)
        sd_expanded = tf.expand_dims(tf.expand_dims(sd_input, -1), -1)
        sd_embed = self.sd_projection(sd_expanded)
        sd_embed = tf.broadcast_to(sd_embed, tf.shape(x))
        x = x + sd_embed
        x = self.dec_dropout(x, training=training)
        for dec_layer in self.decoder_layers_list:
            x = dec_layer(x, enc_output,
                          look_ahead_mask=combined_mask,
                          padding_mask=enc_padding_mask,
                          training=training)
        return x

    def call(self, inputs, training=False):
        enc_input, dec_input, sd_input = inputs
        enc_output = self.encode(enc_input, sd_input, training=training)
        dec_output = self.decode(dec_input, enc_output, enc_input, sd_input,
                                training=training)
        return self.final_layer(dec_output)


def create_model():
    model = TSGPTransformer()
    enc_input = np.zeros((1, TRANSFORMER_MAX_SEQ_LEN), dtype=np.int32)
    dec_input = np.zeros((1, TRANSFORMER_MAX_SEQ_LEN), dtype=np.int32)
    sd_input = np.zeros((1,), dtype=np.float32)
    model([enc_input, dec_input, sd_input], training=False)
    return model

print("Model class defined.")

In [ ]:
# --- Training ---
import glob
import re
import pickle
from tqdm import tqdm


class TSGPTrainingModel(keras.Model):
    def __init__(self, tsgp_model, **kwargs):
        super().__init__(**kwargs)
        self.tsgp_model = tsgp_model

    def call(self, inputs, training=False):
        return self.tsgp_model(inputs, training=training)


def masked_loss(y_true, y_pred):
    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction="none")
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)


def find_latest_checkpoint(checkpoint_dir):
    pattern = os.path.join(checkpoint_dir, "tsgp_epoch_*.weights.h5")
    files = glob.glob(pattern)
    if not files:
        return None, 0
    epoch_numbers = []
    for f in files:
        match = re.search(r"tsgp_epoch_(\d+)\.weights\.h5$", f)
        if match:
            epoch_numbers.append((int(match.group(1)), f))
    if not epoch_numbers:
        return None, 0
    epoch_numbers.sort(key=lambda x: x[0])
    latest_epoch, latest_file = epoch_numbers[-1]
    return latest_file, latest_epoch


def prompt_resume(checkpoint_dir):
    latest_file, latest_epoch = find_latest_checkpoint(checkpoint_dir)
    if latest_file is None:
        return 0, None

    print(f"\nFound existing checkpoint: {latest_file} (epoch {latest_epoch})")
    all_files = sorted(glob.glob(
        os.path.join(checkpoint_dir, "tsgp_epoch_*.weights.h5")))
    print("Available checkpoints:")
    for f in all_files:
        print(f"  {os.path.basename(f)}")

    while True:
        choice = input(
            f"\nResume from epoch {latest_epoch}? [Y/n/epoch_number]: "
        ).strip()
        if choice == "" or choice.lower() == "y":
            return latest_epoch, latest_file
        if choice.lower() == "n":
            return 0, None
        try:
            epoch_num = int(choice)
            target = os.path.join(
                checkpoint_dir, f"tsgp_epoch_{epoch_num}.weights.h5")
            if os.path.exists(target):
                return epoch_num, target
            print(f"  Checkpoint for epoch {epoch_num} not found.")
        except ValueError:
            print("  Enter Y, n, or an epoch number.")


def train_model(data_path, checkpoint_dir="checkpoints",
                epochs=TRANSFORMER_EPOCHS, batch_size=TRANSFORMER_BATCH_SIZE):
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)

    start_epoch, resume_path = prompt_resume(checkpoint_dir)

    print("Loading training data...")
    with open(data_path, "rb") as f:
        training_data = pickle.load(f)
    print(f"Loaded {len(training_data)} training pairs")

    input_tokens = [d["input_tokens"] for d in training_data]
    output_tokens = [d["output_tokens"] for d in training_data]
    sd_values = np.array([d["sd"] for d in training_data], dtype=np.float32)

    print("Encoding sequences...")
    enc_input = batch_encode_encoder(input_tokens)
    dec_full = batch_encode_decoder(output_tokens)
    dec_input = dec_full[:, :-1]
    dec_target = dec_full[:, 1:]

    del training_data, input_tokens, output_tokens, dec_full

    print("Building model...")
    base_model = create_model()

    if resume_path is not None:
        base_model.load_weights(resume_path)
        print(f"Loaded weights from {resume_path}")
        print(f"Resuming training from epoch {start_epoch + 1}")

    wrapper = TSGPTrainingModel(base_model)
    optimizer = keras.optimizers.Adam(learning_rate=TRANSFORMER_LEARNING_RATE)
    wrapper.compile(optimizer=optimizer, loss=masked_loss)

    base_model.summary()

    for epoch in range(start_epoch + 1, epochs + 1):
        n = len(enc_input)
        indices = np.random.permutation(n)
        total_loss = 0.0
        num_batches = 0

        batch_iter = tqdm(
            range(0, n, batch_size),
            desc=f"Epoch {epoch}/{epochs}",
            unit="batch",
            total=(n + batch_size - 1) // batch_size,
        )

        for start in batch_iter:
            end = min(start + batch_size, n)
            idx = indices[start:end]

            loss = wrapper.train_on_batch(
                [enc_input[idx], dec_input[idx], sd_values[idx]],
                dec_target[idx]
            )
            total_loss += float(loss)
            num_batches += 1
            batch_iter.set_postfix(loss=f"{total_loss / num_batches:.4f}")

        avg_loss = total_loss / max(num_batches, 1)
        print(f"Epoch {epoch}/{epochs} — Loss: {avg_loss:.4f}")

        base_model.save_weights(
            os.path.join(checkpoint_dir, f"tsgp_epoch_{epoch}.weights.h5"))

    base_model.save_weights(
        os.path.join(checkpoint_dir, "tsgp_final.weights.h5"))
    print(f"\nTraining complete. Weights saved to {checkpoint_dir}/")
    return base_model

print("Training function defined.")

In [ ]:
# --- Run Training ---
model = train_model("training_pairs.pkl")

In [ ]:
# --- Download trained weights ---
from google.colab import files

files.download("checkpoints/tsgp_final.weights.h5")